# Labo 30 · Carnet Colab — Un CNN Keras

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = x_train[..., np.newaxis].astype("float32") / 255.0      # (n, 28, 28, 1) : le canal à la fin
x_test = x_test[..., np.newaxis].astype("float32") / 255.0
noms = ["t-shirt", "pantalon", "chandail", "robe", "manteau", "sandale", "chemise", "espadrille", "sac", "botte"]
print(x_train.shape, x_test.shape)

## 1. La référence : un perceptron multicouche

In [ ]:
mlp = keras.Sequential([keras.Input(shape=(28, 28, 1)), keras.layers.Flatten(),
                        keras.layers.Dense(256, activation="relu"), keras.layers.Dense(10, activation="softmax")])
mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mlp.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
print("MLP :", mlp.count_params(), "paramètres, exactitude au test", round(mlp.evaluate(x_test, y_test, verbose=0)[1], 4))

## 2. Un CNN

**À toi** : le CNN de l'exemple du cours : `Conv2D(32, 3, padding="same")` et ReLU, `MaxPooling2D(2)`, `Conv2D(64, 3, padding="same")` et ReLU, `MaxPooling2D(2)`, `Flatten`, puis `Dense(10)` softmax. Affiche `summary` (vérifie les 50 186 paramètres), entraîne 5 époques comme le MLP, et mesure le temps de chaque époque (`time.time()` autour de `fit`, divisé par 5).

In [ ]:
# À toi
cnn = None

**Question.** Le CNN a quatre fois moins de paramètres que le MLP. Est-il meilleur? Pourquoi?

## 3. L'augmentation de données

**À toi** : un nouveau CNN, identique, précédé de deux couches d'augmentation, `keras.layers.RandomTranslation(0.1, 0.1)` et `keras.layers.RandomZoom(0.1)` (elles ne s'appliquent qu'à l'entraînement). Entraîne-le 5 époques, puis compare les deux CNN sur une version **décalée** du test : `np.roll(x_test, 4, axis=2)` (4 pixels vers la droite).

In [ ]:
# À toi
cnn_aug = None

## 4. Ce que voit le réseau

Les filtres de la première couche sont dans `cnn.layers[0].get_weights()[0]`, de forme (3, 3, 1, 32). Pour voir les cartes de caractéristiques d'une image, on construit un modèle qui s'arrête après la première convolution.

In [ ]:
filtres = cnn.layers[0].get_weights()[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(filtres[:, :, 0, k], cmap="RdBu"); ax.axis("off")
fig.suptitle("16 des 32 filtres appris"); plt.show()

premiere = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
cartes = premiere.predict(x_test[:1], verbose=0)[0]          # (28, 28, 32)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
axes[0, 0].imshow(x_test[0, :, :, 0], cmap="gray"); axes[0, 0].set_title(noms[y_test[0]])
for k, ax in enumerate(axes.ravel()[1:]):
    ax.imshow(cartes[:, :, k], cmap="viridis")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : le nombre de paramètres et l'exactitude au test du MLP et du CNN, le temps d'une époque du CNN, les résultats sur le test décalé avec et sans augmentation, et une phrase sur ce que montrent les cartes de caractéristiques.